# Document Chunking Strategies: where you cut decides what you can find

Before any text can be embedded, indexed, or retrieved, it has to be **split into chunks** — and *how* you make that cut silently sets the ceiling on everything downstream. A perfect embedder and a perfect LLM cannot recover a fact your chunker sliced in half.

This notebook builds three chunkers from scratch — fixed-size-with-overlap, recursive (structure-aware), and semantic (cut where adjacent-sentence embeddings diverge) — and **measures** which keeps a fact retrievable, reusing the chapter-01 hashing embedder + cosine retrieval so it runs on CPU in milliseconds. We import the canonical functions from [`document_chunking.py`](document_chunking.py); the embedder is **hash-deterministic** (fixed FNV-1a, no random step), so every number below is identical on every run.

**By the end you'll have:** watched a naive fixed cut shatter a fact, built recursive and semantic splitters that keep it whole, *seen* the semantic similarity trace dip at section boundaries, and **measured** how the cut strategy alone decides retrieval success.

## Step 1 — set up: import the canonical chunking functions

Everything below uses the verified functions from `document_chunking.py` — the same code the concept page shows. We print versions (and the torch device if present) and the toy document: a short multi-section mission spec with headings and a launch-date fact that the chunkers will either keep whole or split.

In [1]:
import numpy as np

from document_chunking import (
    DOCUMENT, PROBE_QUESTION, ANSWER_PHRASE, FIXED_CHUNK_CHARS,
    chunk_fixed, chunk_recursive, chunk_semantic,
    compute_idf, adjacent_similarities, evaluate_strategy,
    answer_survives_any_chunk, _split_sentences,
)

print('numpy:', np.__version__)
try:
    import torch
    device = ('cuda' if torch.cuda.is_available()
              else 'mps' if torch.backends.mps.is_available() else 'cpu')
    print('torch:', torch.__version__, '| device:', device, '(chunking + retrieval are pure numpy)')
except ImportError:
    print('torch: not installed (chunking + retrieval are pure numpy — unaffected)')

print(f'\ndocument: {len(DOCUMENT)} chars | probe: {PROBE_QUESTION!r}')
print(DOCUMENT)

numpy: 2.4.6


torch: 2.12.0 | device: mps (chunking + retrieval are pure numpy)

document: 562 chars | probe: 'When was the Helios-7 satellite launched?'
# Helios-7 Mission Overview

The Helios-7 satellite is an Earth-observation platform operated by the Nairobi office.
It was launched on March 3rd, 2024 from the Kourou spaceport aboard an Ariane 6 rocket.

# Instruments

Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
The imager captures 200 spectral bands across the visible and near-infrared range.

# Orbit

Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
This orbit keeps the local solar time of each pass roughly constant for stable imaging.


## Step 2 — watch a naive fixed cut shatter the fact

The simplest strategy is **fixed-size**: cut every N characters, content-blind. With `size=140` the boundary lands at offset 140 — which falls *inside* the launch-date phrase. Run it and check each chunk: the phrase `March 3rd, 2024` is whole in **none** of them. The fact was in the document; the blind cut destroyed it as a retrievable unit.

In [2]:
naive = chunk_fixed(DOCUMENT, size=FIXED_CHUNK_CHARS, overlap=0)
for i, c in enumerate(naive):
    intact = ANSWER_PHRASE in c
    print(f'chunk[{i}] ({len(c):>3} chars) holds the date: {intact}')

# correctness BEFORE any claim: the fact must be intact in NO chunk
assert not answer_survives_any_chunk(naive), 'naive 140-char split should shatter the fact'
print(f'\n{ANSWER_PHRASE!r} intact in some chunk:', answer_survives_any_chunk(naive))

chunk[0] (140 chars) holds the date: False
chunk[1] (140 chars) holds the date: False
chunk[2] (140 chars) holds the date: False
chunk[3] (139 chars) holds the date: False
chunk[4] (  2 chars) holds the date: False

'March 3rd, 2024' intact in some chunk: False


## Step 3 — the recursive splitter keeps ideas whole

The fix is to cut on **natural boundaries**. The recursive splitter tries the coarsest boundary first (paragraphs, split on a blank line) and only descends to sentences when a piece overflows. Run it: cuts land on paragraph/heading boundaries, so the launch-date sentence stays whole inside one chunk — and the fact survives.

In [3]:
recursive = chunk_recursive(DOCUMENT)
for i, c in enumerate(recursive):
    print(f'chunk[{i}] ({len(c):>3} chars): {c[:55]!r}')

assert answer_survives_any_chunk(recursive), 'recursive split must keep the fact whole'
print(f'\n{ANSWER_PHRASE!r} intact in some chunk:', answer_survives_any_chunk(recursive))

chunk[0] ( 27 chars): '# Helios-7 Mission Overview'
chunk[1] (175 chars): 'The Helios-7 satellite is an Earth-observation platform'
chunk[2] ( 13 chars): '# Instruments'
chunk[3] (160 chars): 'Helios-7 carries a hyperspectral imager with a ground r'
chunk[4] (  7 chars): '# Orbit'
chunk[5] (170 chars): 'Helios-7 completes one orbit of Earth every 97 minutes '

'March 3rd, 2024' intact in some chunk: True


## Step 4 — the semantic boundary signal: see the dips

Semantic chunking cuts where the **topic shifts** — where the cosine similarity between adjacent sentences drops. Embed each sentence, measure consecutive-pair similarities, and read the trace: the low values are the section seams. Watch for the dips at the boundaries between the Overview, Instruments, and Orbit sections — those are exactly where a human would cut.

In [4]:
sents = _split_sentences(DOCUMENT)
sims = adjacent_similarities(sents, compute_idf(sents))
print('sentences:', len(sents))
for i, s in enumerate(sims):
    bar = '#' * int(s * 60)
    print(f'  gap {i}|{i+1}: cos={s:.3f}  {bar}')
print('\nthe low bars (gaps 1|2, 2|3) are the section boundaries the splitter cuts on')

sentences: 6
  gap 0|1: cos=0.135  ########
  gap 1|2: cos=0.000  
  gap 2|3: cos=0.048  ##
  gap 3|4: cos=0.075  ####
  gap 4|5: cos=0.145  ########

the low bars (gaps 1|2, 2|3) are the section boundaries the splitter cuts on


## Step 5 — the semantic splitter cuts at the dips

Feed that signal to the splitter: it cuts wherever similarity falls below the 35th percentile of the trace (the LlamaIndex breakpoint-percentile method), grouping the rest into topic-coherent chunks. The result is 3 chunks at the natural section seams — and the launch-date fact stays whole.

In [5]:
idf_sent = compute_idf(_split_sentences(DOCUMENT))
semantic = chunk_semantic(DOCUMENT, idf_sent)
for i, c in enumerate(semantic):
    print(f'chunk[{i}] ({len(c):>3} chars): {c[:60]!r}')

assert answer_survives_any_chunk(semantic), 'semantic split must keep the fact whole'
print(f'\n{len(semantic)} topic-coherent chunks; fact intact:', answer_survives_any_chunk(semantic))

chunk[0] (203 chars): '# Helios-7 Mission Overview The Helios-7 satellite is an Ear'
chunk[1] ( 91 chars): '# Instruments Helios-7 carries a hyperspectral imager with a'
chunk[2] (261 chars): 'The imager captures 200 spectral bands across the visible an'

3 topic-coherent chunks; fact intact: True


## Step 6 — measure all four: the cut strategy decides retrieval

Now the payoff. Index each strategy's chunks, retrieve the top-1 for the probe question, and check whether the answer survived. **Read the table:** naive fixed loses the fact; fixed+overlap recovers it in *some* chunk but the boundary/runt chunks still confuse top-1; recursive and semantic both retrieve it. Same document, same embedder, same query — **only the cut points changed.**

In [6]:
strategies = {
    'fixed (no overlap)': chunk_fixed(DOCUMENT, overlap=0),
    'fixed (+overlap)':   chunk_fixed(DOCUMENT, overlap=40),
    'recursive':          chunk_recursive(DOCUMENT),
    'semantic':           chunk_semantic(DOCUMENT, idf_sent),
}
print(f"{'strategy':<20} | {'#chunks':>7} | {'answer in top-1':>15} | intact anywhere")
print('-' * 70)
for name, chunks in strategies.items():
    res = evaluate_strategy(name, chunks, compute_idf(chunks))
    print(f'{name:<20} | {res.n_chunks:>7} | {str(res.answer_intact):>15} | {answer_survives_any_chunk(chunks)}')

# the headline assertions: blind split fails, structure/meaning-aware splits succeed
assert not evaluate_strategy('a', strategies['fixed (no overlap)'], compute_idf(strategies['fixed (no overlap)'])).answer_intact
assert evaluate_strategy('b', strategies['recursive'], compute_idf(strategies['recursive'])).answer_intact
assert evaluate_strategy('c', strategies['semantic'], compute_idf(strategies['semantic'])).answer_intact
print('\nnaive split loses the fact; recursive & semantic retrieve it: True')

strategy             | #chunks | answer in top-1 | intact anywhere
----------------------------------------------------------------------
fixed (no overlap)   |       5 |           False | False
fixed (+overlap)     |       6 |           False | True
recursive            |       6 |            True | True
semantic             |       3 |            True | True

naive split loses the fact; recursive & semantic retrieve it: True


## Try it yourself

Before you change anything, **predict**: the naive split failed at `size=140` because a boundary landed inside the date (offsets 136–151). If you change the chunk size to **120**, does the fact survive — and does it then get retrieved?

Then try it: `chunk_fixed(DOCUMENT, size=120, overlap=0)` and check `answer_survives_any_chunk`. *Hint:* at size=120 the boundaries land at 120 and 240, **neither inside** 136–151, so the date stays whole this time — which exposes the real problem with fixed-size: whether it works is an **accident of where the ruler happens to land**, not a property you can rely on. That unreliability is exactly why recursive and semantic splitting exist.

> To see chunk boundaries interactively, paste any text into [ChunkViz](https://chunkviz.up.railway.app/) and slide the size/overlap controls — the cuts light up live.

## What we saw

- **A blind fixed cut shattered the fact** — at `size=140` the boundary fell inside `March 3rd, 2024`, leaving it whole in no chunk and unretrievable.
- **Recursive splitting kept ideas whole** — cutting on paragraph/sentence boundaries preserved the fact, so retrieval found it.
- **Semantic splitting cut at topic shifts** — the adjacent-sentence similarity trace dipped at the section seams (gaps 1|2, 2|3), and cutting there gave 3 coherent chunks with the fact intact.
- **The cut strategy alone decided success** — same document, embedder, and query; only the chunk boundaries changed, and that flipped retrieval from failure to success.

**What we skipped** (and where to read it): **overlap's token cost** and the recall/precision curves (on the concept page); **parent-document / sentence-window** retrieval (retrieve small, feed large); **contextual retrieval** (prepend context before embedding); and structure-aware splitting for tables and code. All of that is in the [Document Chunking concept page](../02-Document-Chunking-Strategies.md), with its [curated references](../02-Document-Chunking-Strategies.references.md). Next: better [embeddings](../../03-Embedding-Models-for-Retrieval/03-Embedding-Models-for-Retrieval.md) so paraphrases match — a problem chunking can't fix.